Use T4 GPUs to make it faster

Imports libraries and defines the theme for the table

In [1]:
%pip install qdrant-client sentence-transformers rich
%pip install rich-theme-manager

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 390.4/390.4 kB 7.6 MB/s eta 0:00:00:00:01
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer
from threading import Thread
from rich.console import Console
from rich.panel import Panel
from rich.markdown import Markdown
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer

# --- STEP 2: VISUAL SETUP ---
from rich.style import Style
from rich_theme_manager import Theme, ThemeManager
import pathlib
import pandas as pd
import warnings
import sys

# --- STEP 4: SETUP VECTOR DB ---
from qdrant_client import models, QdrantClient
from sentence_transformers import SentenceTransformer

# Define Theme
THEMES = [
    Theme(
        name="dark",
        description="Dark mode theme",
        tags=["dark"],
        styles={
            "repr.own": Style(color="#e87d3e", bold=True),
            "repr.tag_name": "dim cyan",
            "repr.call": "bright_yellow",
            "repr.str": "bright_green",
            "repr.number": "bright_red",
            "repr.none": "dim white",
            "repr.attrib_name": Style(color="#e87d3e", bold=True),
            "repr.attrib_value": "bright_blue",
            "default": "bright_white on black"
        },
    )
]
theme_dir = pathlib.Path("themes").expanduser()
theme_dir.mkdir(parents=True, exist_ok=True)
theme_manager = ThemeManager(theme_dir=theme_dir, themes=THEMES)
console = Console(theme=theme_manager.get("dark"))

2026-02-19 22:24:14.015064: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771539854.189433      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771539854.241590      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771539854.650759      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771539854.650800      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771539854.650803      55 computation_placer.cc:177] computation placer alr

Initial Setup and Loads Dataset

In [3]:
# --- SETUP (Fast Reload) ---
console = Console()
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "Qwen/Qwen2.5-3B-Instruct"

# --- STEP 3: LOAD DATA ---
warnings.filterwarnings('ignore')
console.print("[bold green]Loading NIST Data...[/bold green]")

try:
    df = pd.read_csv('/kaggle/input/nist-800-53/NIST_SP-800-53.csv')
    df['combined_text'] = (
        "Control ID: " + df['identifier'].astype(str) + "; " +
        "Title: " + df['name'].astype(str) + "; " +
        "Control Text: " + df['control_text'].fillna('').astype(str) + "; " +
        "Discussion: " + df['discussion'].fillna('').astype(str)
    )
    data = df.to_dict('records')
    console.print(f"[dim]Loaded {len(data)} rows successfully.[/dim]")
except FileNotFoundError:
    console.print("[bold red]CRITICAL ERROR: CSV file not found![/bold red]")
    console.print("[yellow]Please re-upload 'NIST_SP-800-53.csv' to the Files panel.[/yellow]")
    data = []

# 1. Load Resources (Only if not already loaded to save time)
if 'qdrant' not in globals():
    console.print("[bold yellow]Reloading Database connection...[/bold yellow]")
    qdrant = QdrantClient(":memory:")
    encoder = SentenceTransformer('all-MiniLM-L6-v2')
    # Re-index data if needed (Assuming 'data' variable exists from previous run)
    if 'data' in globals() and data:
        qdrant.recreate_collection(
            collection_name="nist_controls",
            vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE)
        )
        qdrant.upload_points(
            collection_name="nist_controls",
            points=[
                models.PointStruct(id=idx, vector=encoder.encode(doc["combined_text"]).tolist(), payload=doc)
                for idx, doc in enumerate(data)
            ]
        )

if 'model' not in globals():
    console.print(f"[bold yellow]Loading Model ({device})...[/bold yellow]")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32
    ).to(device)

Loading NIST Data...

CRITICAL ERROR: CSV file not found!

Please re-upload 'NIST_SP-800-53.csv' to the Files panel.

Reloading Database connection...

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Chat Loop

In [4]:
# --- MAIN CHAT LOOP ---
console.print("\n[bold green]✅ System Ready! Type 'exit' or 'quit' to stop.[/bold green]")

while True:
    # 1. Get Question
    console.print("\n[bold cyan]Your Question:[/bold cyan]")
    user_prompt = input(">>> ")

    if user_prompt.lower() in ["exit", "quit", "stop"]:
        console.print("[bold red]Stopping program. Goodbye![/bold red]")
        break

    if not user_prompt.strip():
        continue

    # 2. Search Database
    query_vector = encoder.encode(user_prompt).tolist()
    try:
        hits = qdrant.search(collection_name="nist_controls", query_vector=query_vector, limit=10)
    except AttributeError:
        # Fallback for different versions
        hits = qdrant.query_points(collection_name="nist_controls", query=query_vector, limit=10).points

    # 3. Prepare Context
    context_text = "\n".join([
        f"- ID: {hit.payload.get('identifier')} | Title: {hit.payload.get('name')} | Text: {hit.payload.get('control_text')}"
        for hit in hits
    ])

    # Display Results Table
    from rich.table import Table
    table = Table(title="Relevant NIST 800-53 Controls", show_lines=True)
    table.add_column("Control ID", style="bright_red")
    table.add_column("Title", style="green")
    table.add_column("Snippet", style="yellow")
    table.add_column("Score", style="#a6accd")

    for hit in hits:
        snippet = str(hit.payload.get("control_text", ""))[:150] + "..."
        table.add_row(
            str(hit.payload.get("identifier", "N/A")),
            str(hit.payload.get("name", "N/A")),
            snippet,
            f"{hit.score:.4f}"
        )
    console.print(table)

    # 4. Stream Answer
    messages = [
        {"role": "system", "content": "You are a NIST SP 800-53 expert. Answer the user based strictly on the context provided."},
        {"role": "user", "content": f"Context:\n{context_text}\n\nQuestion: {user_prompt}\n\nAnswer:"}
    ]

    # Get the input IDs
    input_ids = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(device)
    
    # Create the attention_mask (all 1s, same shape as input_ids)
    attention_mask = torch.ones_like(input_ids)

    # Initialize Streamer
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    
    # Pass BOTH input_ids and attention_mask to the generator
    generation_kwargs = dict(
        input_ids=input_ids, 
        attention_mask=attention_mask, 
        streamer=streamer, 
        max_new_tokens=400
    )

    # Run generation in a separate thread
    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    console.print("\n[bold green]NIST Recommendation:[/bold green]")

    # Print tokens as they arrive
    generated_text = ""
    for new_text in streamer:
        print(new_text, end="", flush=True)
        generated_text += new_text
    print() # Newline at end

✅ System Ready! Type 'exit' or 'quit' to stop.

Your Question:

>>>  exit


Stopping program. Goodbye!

In [6]:
import gradio as gr
import pandas as pd

# 1. Wrap your original logic in a function
def nist_rag_chat(user_prompt, history):
    # --- ORIGINAL SEARCH LOGIC ---
    query_vector = encoder.encode(user_prompt).tolist()
    try:
        hits = qdrant.search(collection_name="nist_controls", query_vector=query_vector, limit=5)
    except:
        hits = qdrant.query_points(collection_name="nist_controls", query=query_vector, limit=5).points

    # --- ORIGINAL CONTEXT PREP ---
    context_text = "\n".join([
        f"- ID: {hit.payload.get('identifier')} | Title: {hit.payload.get('name')} | Text: {hit.payload.get('control_text')}"
        for hit in hits
    ])
    
    # Create the Table Data for the UI
    table_data = []
    for hit in hits:
        table_data.append([
            hit.payload.get("identifier", "N/A"),
            hit.payload.get("name", "N/A"),
            hit.payload.get("control_text", "")[:200] + "...", # Snippet for the table
            round(hit.score, 4)
        ])
    df_results = pd.DataFrame(table_data, columns=["Control ID", "Title", "Snippet", "Score"])

    # --- ORIGINAL GENERATION LOGIC ---
    messages = [
        {"role": "system", "content": "You are a NIST SP 800-53 expert. Answer based strictly on the context provided."},
        {"role": "user", "content": f"Context:\n{context_text}\n\nQuestion: {user_prompt}\n\nAnswer:"}
    ]

    input_ids = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(device)
    attention_mask = torch.ones_like(input_ids)
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    
    generation_kwargs = dict(input_ids=input_ids, attention_mask=attention_mask, streamer=streamer, max_new_tokens=400)
    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    # Yield the table immediately, then stream the text
    response_text = ""
    for new_token in streamer:
        response_text += new_token
        # Return the current text and the table of search results
        yield response_text, df_results

# --- THE UI LAYOUT ---
with gr.Blocks(theme="soft", title="NIST RAG Expert") as demo:
    gr.Markdown("# 🛡️ NIST SP 800-53 Expert")
    
    with gr.Row():
        with gr.Column(scale=2):
            # The Chat Interface
            chatbot = gr.Chatbot(label="NIST Advisor", height=500, type="messages")
            msg = gr.Textbox(placeholder="Ask about a control (e.g., AC-2)...", label="Your Question")
            submit = gr.Button("Submit", variant="primary")
            
        with gr.Column(scale=1):
            # The Results Table
            gr.Markdown("### 🔍 Supporting Evidence")
            results_table = gr.Dataframe(
                headers=["ID", "Title", "Snippet", "Score"],
                datatype=["str", "str", "str", "number"],
                label="Retrieved Controls",
                interactive=False
            )

    # State for tracking history internally
    history_state = gr.State([])

    # Link everything together
    def user_action(user_msg, history):
        return "", history + [{"role": "user", "content": user_msg}]

    def bot_action(history):
        user_msg = history[-1]["content"]
        # The function yields (text, table)
        for text, table in nist_rag_chat(user_msg, history):
            history[-1]["content"] = text
            yield history, table

    # Trigger flow
    msg.submit(user_action, [msg, history_state], [msg, history_state]).then(
        bot_action, [history_state], [chatbot, results_table]
    )
    submit.click(user_action, [msg, history_state], [msg, history_state]).then(
        bot_action, [history_state], [chatbot, results_table]
    )

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://2a7b1ab063c385d33e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
